In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import heapq  
import pickle
from string import punctuation
from nltk.corpus import stopwords
punctuation = punctuation + '\n'
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
from google.colab import files


uploaded = files.upload()


Saving summarizdataset.csv to summarizdataset.csv


In [ ]:
import pandas as pd
import io
 
df = pd.read_csv(io.BytesIO(uploaded['summarizdataset.csv']))
print(df)

                                                   text       type  \
0     \nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...    culture   
1     \nتحصل كتاب "المصحف وقراءاته" الذي ألفه باحثون...    culture   
2     \nاحتضن جناح تونس في القرية الدولية للأفلام بم...    culture   
3     \nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...    culture   
4     \nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...    culture   
...                                                 ...        ...   
8373  \nتأجل الإضراب العام في قطاع الصحة الذي كان مق...  localnews   
8374  \nكشف الناشطان كريم نوار وعفيف زقية اللذان أشر...  localnews   
8375  \nتمكّنت فرقة الأبحاث والتفتيش للحرس الوطني بط...  localnews   
8376  \nقرر الأهالي بمناطق هيشر وعين القارصي والغولا...  localnews   
8377  \nتمكنت وحدات الحرس الوطني بمحطة الإستخلاص ببر...  localnews   

                                         Processed Text  \
0     اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...   
1     تحصل كتاب المصحف وقراءاته الفه باحث

In [ ]:
def nltk_summarizer(input_text):
    number_of_sentence = 3
    stopWords = set(nltk.corpus.stopwords.words("arabic") + nltk.corpus.stopwords.words("english"))
    word_frequencies = {}  
    for word in nltk.word_tokenize(input_text):  
        if word not in stopWords:
            if word not in punctuation:
                if word not in word_frequencies.keys():
                    word_frequencies[word] = 1
                else:
                    word_frequencies[word] += 1

    maximum_frequncy = max(list(word_frequencies.values()),default=3)

    for word in word_frequencies.keys():  
        word_frequencies[word] = (word_frequencies[word]/maximum_frequncy)

    sentence_list = nltk.sent_tokenize(input_text)
    sentence_scores = {}  
    for sent in sentence_list:
        for word in nltk.word_tokenize(sent.lower()):
            if word in word_frequencies.keys():
                if len(sent.split(' ')) < 30:
                    if sent not in sentence_scores.keys():
                        sentence_scores[sent] = word_frequencies[word]
                    else:
                        sentence_scores[sent] += word_frequencies[word]

    summary_sentences = heapq.nlargest(number_of_sentence, sentence_scores, key=sentence_scores.get)

    summary = ' '.join(summary_sentences)  
    return summary

In [ ]:
def delete_links(input_text):
    pettern  = r'''(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'".,<>?«»“”‘’]))'''
    out_text = re.sub(pettern, ' ', input_text)
    return out_text

def delete_repeated_characters(input_text):
    pattern  = r'(.)\1{2,}'
    out_text = re.sub(pattern, r"\1\1", input_text)
    return out_text

def replace_letters(input_text):
    replace = {"أ": "ا","ة": "ه","إ": "ا","آ": "ا","": ""}
    replace = dict((re.escape(k), v) for k, v in replace.items()) 
    pattern = re.compile("|".join(replace.keys()))
    out_text = pattern.sub(lambda m: replace[re.escape(m.group(0))], input_text)
    return out_text

def clean_text(input_text):
    replace = r'[/(){}\[\]|@âÂ,;\?\'\"\*…؟–’،!&\+-:؛-]'
    out_text = re.sub(replace, " ", input_text)
    words = nltk.word_tokenize(out_text)
    words = [word for word in words if word.isalpha()]
    out_text = ' '.join(words)
    return out_text

def remove_vowelization(input_text):
    vowelization = re.compile(""" ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ""", re.VERBOSE)
    out_text = re.sub(vowelization, '', input_text)
    return out_text

def delete_stopwords(input_text):
    stop_words = set(nltk.corpus.stopwords.words("arabic") + nltk.corpus.stopwords.words("english"))
    tokenizer = nltk.tokenize.WhitespaceTokenizer()
    tokens = tokenizer.tokenize(input_text)
    wnl = nltk.WordNetLemmatizer()
    lemmatizedTokens =[wnl.lemmatize(t) for t in tokens]
    out_text = [w for w in lemmatizedTokens if not w in stop_words]
    out_text = ' '.join(out_text)
    return out_text

def stem_text(input_text):
    st = ISRIStemmer()
    tokenizer = nltk.tokenize.WhitespaceTokenizer()
    tokens = tokenizer.tokenize(input_text)
    out_text = [st.stem(w) for w in tokens]
    out_text = ' '.join(out_text)
    return out_text


def text_prepare(input_text, ar_text):
    out_text = delete_links(input_text)
    out_text = delete_repeated_characters(out_text)
    out_text = clean_text(out_text)
    out_text = delete_stopwords(out_text)
    if ar_text:
        out_text = replace_letters(out_text)
        out_text = remove_vowelization(out_text)
    else:
        out_text = out_text.lower()
    return out_text

In [ ]:
df['Processed Text'] = df['text'].apply(text_prepare, args=(True,))

In [ ]:
df

,text,type,Processed Text,summarizer
0,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,culture,اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...",culture,تحصل كتاب المصحف وقراءاته الفه باحثون تونسيون ...,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,\nاحتضن جناح تونس في القرية الدولية للأفلام بم...,culture,احتضن جناح تونس القريه الدوليه للافلام بمدينه ...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,\nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...,culture,شهدت برلين الجمعه افتتاح مسجد فريد نوعه الاقل ...,واستأجرت صاحبة المشروع المحامية والكاتبة سيران...
4,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,culture,نعت وزاره المنشد عز بن محمود انتقل جوار يوم تن...,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...
...,...,...,...,...
8373,\nتأجل الإضراب العام في قطاع الصحة الذي كان مق...,localnews,تاجل الاضراب العام قطاع الصحه مقررا تنفيذه الي...,وكانت انعقدت عشية أمس في وزارة الشؤون الاجتماع...
8374,\nكشف الناشطان كريم نوار وعفيف زقية اللذان أشر...,localnews,كشف الناشطان كريم نوار وعفيف زقيه اشرفا عمليه ...,وأكدا ان تكلفة العلم لم تتجاوز 3 آلاف دينار مق...
8375,\nتمكّنت فرقة الأبحاث والتفتيش للحرس الوطني بط...,localnews,فرقه الابحاث والتفتيش للحرس الوطني بطبلبه ولاي...,كما تمّ إلقاء القبض على عنصر رابع (عمره 32 سنة...
8376,\nقرر الأهالي بمناطق هيشر وعين القارصي والغولا...,localnews,قرر الاهالي بمناطق هيشر وعين القارصي والغولايث...,وتأتي هذه الخطوة على خلفية ما اعتبره أهالي هذه...


In [ ]:
nltk_summarizer
df['summarizer'] = df['text'].apply(nltk_summarizer)

In [ ]:
df

,text,type,Processed Text,summarizer
0,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,culture,اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...",culture,تحصل كتاب المصحف وقراءاته الفه باحثون تونسيون ...,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,\nاحتضن جناح تونس في القرية الدولية للأفلام بم...,culture,احتضن جناح تونس القريه الدوليه للافلام بمدينه ...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,\nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...,culture,شهدت برلين الجمعه افتتاح مسجد فريد نوعه الاقل ...,واستأجرت صاحبة المشروع المحامية والكاتبة سيران...
4,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,culture,نعت وزاره المنشد عز بن محمود انتقل جوار يوم تن...,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...
...,...,...,...,...
8373,\nتأجل الإضراب العام في قطاع الصحة الذي كان مق...,localnews,تاجل الاضراب العام قطاع الصحه مقررا تنفيذه الي...,وكانت انعقدت عشية أمس في وزارة الشؤون الاجتماع...
8374,\nكشف الناشطان كريم نوار وعفيف زقية اللذان أشر...,localnews,كشف الناشطان كريم نوار وعفيف زقيه اشرفا عمليه ...,وأكدا ان تكلفة العلم لم تتجاوز 3 آلاف دينار مق...
8375,\nتمكّنت فرقة الأبحاث والتفتيش للحرس الوطني بط...,localnews,فرقه الابحاث والتفتيش للحرس الوطني بطبلبه ولاي...,كما تمّ إلقاء القبض على عنصر رابع (عمره 32 سنة...
8376,\nقرر الأهالي بمناطق هيشر وعين القارصي والغولا...,localnews,قرر الاهالي بمناطق هيشر وعين القارصي والغولايث...,وتأتي هذه الخطوة على خلفية ما اعتبره أهالي هذه...


In [ ]:
print(df['summarizer'][100])
print('==========================================')
print(df['text'][100])


أصيبت سائحة فرنسية بجراح بجراح خطيرة جراء "عضة" تلقتها من تمساح حاولت التقاط صورة "سيلفي" الى جانبه أثناء رحلة لها إلى تايلندا. وتعرضت السائحة الفرنسية "بينتلير ليسفيلور"46 عاما، لهذا الحادث بصحبة زوجها داخل حديقة "خو ياي" بوسط تايلاند. وتجاهلت "ليسفيلور"لافتات التحذير التي تشدد على ضرورة عدم الإقتراب من البحيرة، لكنها عندما رأت التمساح، اقتربت نحوه لتحصل على الصورة التي تريدها.

أصيبت سائحة فرنسية بجراح بجراح خطيرة جراء "عضة" تلقتها من تمساح حاولت التقاط صورة "سيلفي" الى جانبه أثناء رحلة لها إلى تايلندا.
وتعرضت السائحة الفرنسية "بينتلير ليسفيلور"46 عاما، لهذا الحادث بصحبة زوجها داخل حديقة "خو ياي" بوسط تايلاند. وتجاهلت "ليسفيلور"لافتات التحذير التي تشدد على ضرورة عدم الإقتراب من البحيرة، لكنها عندما رأت التمساح، اقتربت نحوه لتحصل على الصورة التي تريدها. ونقل أمن الحديقة السائحة المصابة إلى المستشفى فورا حيث تتلقى العلاج الآن من إصابتها.



In [ ]:
def aboutsummariz(text:str):
    if len(text)<10:
        ans = 'No'
    else:
        ans = "Yes"
    return ans

In [ ]:
df['done?'] = df['summarizer'].apply(aboutsummariz)

In [ ]:
df

,text,type,Processed Text,summarizer,done?
0,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,culture,اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,Yes
1,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...",culture,تحصل كتاب المصحف وقراءاته الفه باحثون تونسيون ...,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...",Yes
2,\nاحتضن جناح تونس في القرية الدولية للأفلام بم...,culture,احتضن جناح تونس القريه الدوليه للافلام بمدينه ...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...,Yes
3,\nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...,culture,شهدت برلين الجمعه افتتاح مسجد فريد نوعه الاقل ...,واستأجرت صاحبة المشروع المحامية والكاتبة سيران...,Yes
4,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,culture,نعت وزاره المنشد عز بن محمود انتقل جوار يوم تن...,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,Yes
...,...,...,...,...,...
8373,\nتأجل الإضراب العام في قطاع الصحة الذي كان مق...,localnews,تاجل الاضراب العام قطاع الصحه مقررا تنفيذه الي...,وكانت انعقدت عشية أمس في وزارة الشؤون الاجتماع...,Yes
8374,\nكشف الناشطان كريم نوار وعفيف زقية اللذان أشر...,localnews,كشف الناشطان كريم نوار وعفيف زقيه اشرفا عمليه ...,وأكدا ان تكلفة العلم لم تتجاوز 3 آلاف دينار مق...,Yes
8375,\nتمكّنت فرقة الأبحاث والتفتيش للحرس الوطني بط...,localnews,فرقه الابحاث والتفتيش للحرس الوطني بطبلبه ولاي...,كما تمّ إلقاء القبض على عنصر رابع (عمره 32 سنة...,Yes
8376,\nقرر الأهالي بمناطق هيشر وعين القارصي والغولا...,localnews,قرر الاهالي بمناطق هيشر وعين القارصي والغولايث...,وتأتي هذه الخطوة على خلفية ما اعتبره أهالي هذه...,Yes


In [ ]:
import plotly.graph_objects as go

val = dict(df['done?'].value_counts())
labels = list(val.keys())
values = list(val.values())
fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])

fig.show()

In [ ]:
data = df[df['done?']=='Yes']
final = data.drop(['text','done?'],axis=1)

In [ ]:
final.to_csv('summarizdataset.csv',index = False)